# 第 7 章 · 多 agent（Sequential Handoffs）

两个 agent 排成流水线：前一个把结果写进 `session.state`，后一个从 state 里读。

`SequentialAgent` 本身不做推理，它只负责**按顺序调用**。数据不是靠返回值传的，是靠 **state 的键**。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】定位仓库根；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, quiet_adk_sync_warnings, retry_call
quiet_adk_sync_warnings()

from google.adk.runners import InMemoryRunner
from google.genai import types


def run_once(agent, message):
    """跑一次，返回 (每个 agent 的最终答复, 结束时 session.state)。"""
    runner = InMemoryRunner(agent)
    sid = str(uuid.uuid4())
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    finals = []
    for ev in retry_call(lambda: list(runner.run(
            user_id="u", session_id=sid,
            new_message=types.Content(role="user", parts=[types.Part(text=message)]))), label="run "):
        if ev.is_final_response() and ev.content:
            t = getattr(ev.content, "text", None) or "".join(
                p.text for p in (ev.content.parts or []) if getattr(p, "text", None))
            finals.append((getattr(ev, "author", "?"), (t or "").strip()))
    sess = runner.session_service.get_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    return finals, sess.state

from google.adk.agents import LlmAgent, SequentialAgent

# ★ 本章新增 output_key：这个 agent 的最终答复会写进 state[name]
# 【适配】书里两个 agent 都没给 model / instruction，构造出来跑不了
step1 = LlmAgent(
    name="Step1_Fetch",
    model=adk_model(),
    instruction="给出三条关于用户主题的简短事实，每行一条。",
    output_key="data",
)

# 【适配】书里写 state['data']；ADK 只替换 {data} 占位符，那样写模型看不到数据
step2 = LlmAgent(
    name="Step2_Process",
    model=adk_model(),
    instruction="下面是上一步查到的资料：\n{data}\n\n请用一段话总结它。",
)

pipeline = SequentialAgent(name="MyPipeline", sub_agents=[step1, step2])

finals, state = run_once(pipeline, "量子计算")
for author, text in finals:
    print(f"\n--- [{author}] ---\n{text[:400]}")
print("\n--- state['data']（第 1 步的产物）---")
print((state.get("data") or "(空)")[:300])


--- [Step1_Fetch] ---
量子计算利用量子比特的叠加和纠缠特性进行并行运算。
量子计算机在处理特定复杂算法时具有远超传统超级计算机的潜力。
目前量子计算仍处于“含噪声中等规模量子”（NISQ）发展阶段。

--- [Step2_Process] ---
量子计算通过利用量子比特的叠加与纠缠特性实现高效并行运算，在处理特定复杂算法时展现出远超传统超级计算机的巨大潜力，不过目前该技术整体仍处于“含噪声中等规模量子”（NISQ）的发展阶段。

--- state['data']（第 1 步的产物）---
量子计算利用量子比特的叠加和纠缠特性进行并行运算。
量子计算机在处理特定复杂算法时具有远超传统超级计算机的潜力。
目前量子计算仍处于“含噪声中等规模量子”（NISQ）发展阶段。
